**Price Gap Moments Analysis**

This notebook reads in price gap, tariff, and gravity data for multiple years, combines them, and reports correlations between key variables.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

In [ ]:
# Define years and read in data
years = ["2004", "2011", "2017"]
all_dfs = []

# Read gravity data (same for all years)
grav_df = pd.read_csv("../data/top30_gravity_data.csv")

for year in years:
    # Read price gap data
    df = pd.read_csv(f"../data/pricegap-df-{year}.csv")
    df = df.rename(columns={"exporter": "iso_o", "importer": "iso_d"})
    
    # Read tariff data
    tariffs_df = pd.read_csv(f"../data/tariffs-{year}.csv")
    tariffs_df = tariffs_df.rename(columns={"exporter": "iso_o", "importer": "iso_d"})
    
    # Merge price gap with tariffs
    df = df.merge(tariffs_df, on=["iso_o", "iso_d"], how="inner")
    
    # Merge with gravity data
    df = df.merge(grav_df, on=["iso_o", "iso_d"], how="inner")
    
    # Add year column
    df["year"] = year
    
    all_dfs.append(df)
    print(f"Year {year}: {len(df)} observations")

# Combine all years
big_df = pd.concat(all_dfs, ignore_index=True)
print(f"\nTotal observations (all years): {len(big_df)}")

In [ ]:
# Filter out extreme trade share values (Xni ≈ 0 or Xni ≈ 1)
big_df = big_df[~np.isclose(big_df["Xni"], 1.0)]
big_df = big_df[~np.isclose(big_df["Xni"], 0.0)]

print(f"Observations after filtering: {len(big_df)}")
big_df.head()

In [ ]:
# Function to compute correlation with confidence interval
def correlation_with_ci(x, y, name_x, name_y, alpha=0.10):
    """Compute Pearson correlation with confidence interval."""
    # Remove any NaN/Inf values
    mask = np.isfinite(x) & np.isfinite(y)
    x_clean = x[mask]
    y_clean = y[mask]
    
    n = len(x_clean)
    r, p_value = stats.pearsonr(x_clean, y_clean)
    
    # Fisher z-transformation for confidence interval
    z = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)
    z_crit = stats.norm.ppf(1 - alpha/2)
    z_lo, z_hi = z - z_crit * se, z + z_crit * se
    ci_lo, ci_hi = np.tanh(z_lo), np.tanh(z_hi)
    
    print(f"Correlation of {name_x} and {name_y}")
    print(f"  Pearson r = {r:.4f}")
    print(f"  p-value = {p_value:.4e}")
    print(f"  n = {n}")
    print(f"  {int((1-alpha)*100)}% CI: [{ci_lo:.4f}, {ci_hi:.4f}]")
    print()

---
### Summary Statistics

In [ ]:
# Summary statistics for dni by year
dni_summary = []

for year in ["All"] + years:
    if year == "All":
        df_subset = big_df
    else:
        df_subset = big_df[big_df["year"] == year]
    
    dni_summary.append({
        "Year": year,
        "N": len(df_subset),
        "Mean": df_subset["dni"].mean(),
        "Median": df_subset["dni"].median(),
        "Min": df_subset["dni"].min(),
        "Max": df_subset["dni"].max(),
        "Std": df_subset["dni"].std()
    })

dni_stats_df = pd.DataFrame(dni_summary)
print("Summary Statistics for dni")
dni_stats_df

In [ ]:
# Summary statistics for dni2 by year
dni2_summary = []

for year in ["All"] + years:
    if year == "All":
        df_subset = big_df
    else:
        df_subset = big_df[big_df["year"] == year]
    
    dni2_summary.append({
        "Year": year,
        "N": len(df_subset),
        "Mean": df_subset["dni2"].mean(),
        "Median": df_subset["dni2"].median(),
        "Min": df_subset["dni2"].min(),
        "Max": df_subset["dni2"].max(),
        "Std": df_subset["dni2"].std()
    })

dni2_stats_df = pd.DataFrame(dni2_summary)
print("Summary Statistics for dni2")
dni2_stats_df

---
### Summary Correlation Table

In [ ]:
# Build a summary table of correlations by year (using log(dni))
summary_data = []

for year in ["All"] + years:
    if year == "All":
        df_subset = big_df
    else:
        df_subset = big_df[big_df["year"] == year]
    
    # Calculate correlations with p-values using log(dni)
    r_dist, p_dist = stats.pearsonr(np.log(df_subset["dist"]), np.log(df_subset["dni"]))
    r_border, p_border = stats.pearsonr(df_subset["border"], np.log(df_subset["dni"]))
    r_tariff, p_tariff = stats.pearsonr(np.log(1.0 + 0.01 * df_subset["tariff"]), np.log(df_subset["dni"]))
    
    summary_data.append({
        "Year": year,
        "N": len(df_subset),
        "Corr(log(dni), log(dist))": r_dist,
        "p-value (dist)": p_dist,
        "Corr(log(dni), border)": r_border,
        "p-value (border)": p_border,
        "Corr(log(dni), log(1+tariff))": r_tariff,
        "p-value (tariff)": p_tariff
    })

summary_df = pd.DataFrame(summary_data)
pd.set_option('display.float_format', '{:.4f}'.format)
summary_df

In [ ]:
# Build a summary table of correlations by year using log(dni2)
summary_data_dni2 = []

for year in ["All"] + years:
    if year == "All":
        df_subset = big_df
    else:
        df_subset = big_df[big_df["year"] == year]
    
    # Calculate correlations with p-values using log(dni2)
    r_dist, p_dist = stats.pearsonr(np.log(df_subset["dist"]), np.log(df_subset["dni2"]))
    r_border, p_border = stats.pearsonr(df_subset["border"], np.log(df_subset["dni2"]))
    r_tariff, p_tariff = stats.pearsonr(np.log(1.0 + 0.01 * df_subset["tariff"]), np.log(df_subset["dni2"]))
    
    summary_data_dni2.append({
        "Year": year,
        "N": len(df_subset),
        "Corr(log(dni2), log(dist))": r_dist,
        "p-value (dist)": p_dist,
        "Corr(log(dni2), border)": r_border,
        "p-value (border)": p_border,
        "Corr(log(dni2), log(1+tariff))": r_tariff,
        "p-value (tariff)": p_tariff
    })

summary_df_dni2 = pd.DataFrame(summary_data_dni2)
summary_df_dni2